In [50]:
import pandas as pd
import numpy as np

### Salaires (Data)

In [ ]:
def process_compensation_sheet(file_path, sheet_name, country):
    # Charger l'onglet sans header pour pouvoir manipuler les lignes manuellement
    raw_df = pd.read_excel(file_path, sheet_name=sheet_name, header=None)
    
    # Définition des blocs de colonnes (basé sur ton image)
    # Bloc 1 (2023) : col A à D (0:4)
    # Bloc 2 (2024) : col G à J (6:10)
    # Bloc 3 (2025) : col M à P (12:16)
    column_blocks = {
        'Mars 2023': (0, 4),
        'Mars 2024': (6, 10),
        'Mars 2025': (12, 16)
    }
    
    final_frames = []
    
    for period, (start_col, end_col) in column_blocks.items():
        # Extraire le bloc de colonnes
        block = raw_df.iloc[:, start_col:end_col].copy()
        
        # Définir la ligne 4 (index 3 en Python) comme header
        block.columns = block.iloc[3]
        
        # Garder uniquement les données sous le header (ligne 5 et suivantes)
        block = block.iloc[4:].reset_index(drop=True)
        
        # Nettoyage : supprimer les lignes où le libellé de poste est vide
        block = block.dropna(subset=['Libellé de poste'])
        
        cols_to_fix = ['Moyenne de Salaire fixe', 'Moyenne de Variable', 'Effectif']
        for col in cols_to_fix:
            # pd.to_numeric avec errors='coerce' gère les éventuels espaces ou cellules vides
            # .fillna(0) remplace les vides par 0 pour permettre la conversion en int
            block[col] = pd.to_numeric(block[col], errors='coerce').fillna(0).astype(int)
        
        block['Total_Comp_Indiv'] = block['Moyenne de Salaire fixe'] + block['Moyenne de Variable']
        
        # Ajouter les colonnes de contexte
        block['Période'] = period
        block['Country'] = country
        
        final_frames.append(block)
    
    # Fusionner les 3 périodes
    return pd.concat(final_frames, ignore_index=True)


In [19]:
# Application pour les deux pays
df_fr = process_compensation_sheet("Data.xlsx", "Compensation Data FR", "France")
df_lu = process_compensation_sheet("Data.xlsx", "Compensation Data LU", "Luxembourg")
df_data_salaire = pd.concat([df_fr, df_lu], ignore_index=True)
# Afficher les premières lignes du DataFrame final
print(df_data_salaire.sample(15))

3                 Libellé de poste  Effectif  Moyenne de Salaire fixe  \
120  SENIOR PUBLIC AFFAIRS ADVISOR         1                   118000   
689                 TECHNICAL LEAD        11                   104740   
646           MARKET RISKS ANALYST         3                    55364   
333           ENTERPRISE ARCHITECT         1                    69750   
157        CHIEF LICENSING OFFICER         1                    73963   
746           ENTERPRISE ARCHITECT         3                   113254   
546                      ASSISTANT        10                    80777   
813      SENIOR BANKING ACCOUNTANT         1                   107906   
703             BANKING ACCOUNTANT        10                    87665   
399          REGULATORY ACCOUNTANT         2                    53752   
623      HEAD OF REGIONAL COVERAGE         1                   238235   
344           FUND FINANCE OFFICER         2                    70866   
51                FINANCIAL LAWYER        13       

In [20]:
# Afficher toutes les lignes (attention si tu as 10 000 lignes !)
pd.set_option('display.max_rows', None)

# Afficher toutes les colonnes
pd.set_option('display.max_columns', None)

# Ne pas tronquer le contenu des cellules (utile pour les longs libellés de postes)
pd.set_option('display.max_colwidth', None)

In [21]:
print("--- 1. STRUCTURE ET NETTOYAGE ---")
# Types de données et dimensions
print(f"Dimensions : {df_data_salaire.shape[0]} lignes, {df_data_salaire.shape[1]} colonnes")
print("\nTypes de colonnes :")
print(df_data_salaire.dtypes)
    
# Analyse des valeurs manquantes
print("\nValeurs manquantes par colonne :")
print(df_data_salaire.isnull().sum())

--- 1. STRUCTURE ET NETTOYAGE ---
Dimensions : 837 lignes, 7 colonnes

Types de colonnes :
3
Libellé de poste           object
Effectif                    int64
Moyenne de Salaire fixe     int64
Moyenne de Variable         int64
Total_Comp_Indiv            int64
Période                    object
Country                    object
dtype: object

Valeurs manquantes par colonne :
3
Libellé de poste           0
Effectif                   0
Moyenne de Salaire fixe    0
Moyenne de Variable        0
Total_Comp_Indiv           0
Période                    0
Country                    0
dtype: int64


In [23]:
print("\n--- 2. STATISTIQUES DESCRIPTIVES ---")
# Describe sur les colonnes numériques
display(df_data_salaire[['Effectif', 'Moyenne de Salaire fixe', 'Moyenne de Variable', 'Total_Comp_Indiv']].describe())
    


--- 2. STATISTIQUES DESCRIPTIVES ---


3,Effectif,Moyenne de Salaire fixe,Moyenne de Variable,Total_Comp_Indiv
count,837.000000,837.000000,837.000000,837.000000
mean,24.702509,86880.898447,19225.151732,106106.050179
std,149.704114,45107.249395,43557.844748,85438.734145
min,1.000000,38989.000000,0.000000,40236.000000
25%,1.000000,57812.000000,3192.000000,62350.000000
50%,4.000000,73877.000000,5788.000000,80667.000000
75%,11.000000,101000.000000,13750.000000,116642.000000
max,1970.000000,365000.000000,495000.000000,842899.000000


In [26]:
print("\n--- 3. TOP ANALYSES (Période la plus récente) ---")
# On se concentre sur Mars 2025 pour l'analyse actuelle
latest_df = df_data_salaire[df_data_salaire['Période'] == 'Mars 2025'].copy()

# EXCLUSION DES LIGNES DE TOTAL (Crucial pour tes graphs)
# On enlève les lignes où le libellé est 'Total général' ou contient 'Total'
latest_df = latest_df[~latest_df['Libellé de poste'].str.contains('Total', case=False, na=False)]
    
# A. Les postes les mieux rémunérés individuellement (Attractivité)
top_paid_indiv = latest_df.sort_values(by='Total_Comp_Indiv', ascending=False).head(5)
print("\nTop 5 des postes les mieux payés (Salaire + Var moyen) :")
display(top_paid_indiv[['Libellé de poste', 'Country', 'Total_Comp_Indiv', 'Période']])
    
# B. Les postes qui coûtent le plus cher au total (Poids Budgétaire)
# Formule : Total_Comp_Indiv * Effectif
latest_df['Budget_Total_Poste'] = latest_df['Total_Comp_Indiv'] * latest_df['Effectif']
top_budget_postes = latest_df.sort_values(by='Budget_Total_Poste', ascending=False).head(5)
print("\nTop 5 des postes pesant le plus sur la masse salariale :")
display(top_budget_postes[['Libellé de poste', 'Country', 'Effectif', 'Budget_Total_Poste', 'Période']])
    
# C. Les postes avec le plus gros pourcentage de variable (Levier Performance)
# Formule : Variable / Total_Comp_Indiv
latest_df['Pct_Variable'] = (latest_df['Moyenne de Variable'] / latest_df['Total_Comp_Indiv']) * 100
top_variable_pct = latest_df.sort_values(by='Pct_Variable', ascending=False).head(5)
print("\nTop 5 des postes avec la plus forte part de variable (%) :")
display(top_variable_pct[['Libellé de poste', 'Country', 'Pct_Variable', 'Période']])




--- 3. TOP ANALYSES (Période la plus récente) ---

Top 5 des postes les mieux payés (Salaire + Var moyen) :


3,Libellé de poste,Country,Total_Comp_Indiv,Période
732,Deputy CEO,Luxembourg,842899,Mars 2025
323,DEPUTY CHIEF EXECUTIVE OFFICER,France,730000,Mars 2025
733,DEPUTY GENERAL MANAGER,Luxembourg,658833,Mars 2025
723,COUNTRY MANAGING DIRECTOR,Luxembourg,513515,Mars 2025
300,CHIEF HUMAN RESOURCES OFFICER,France,445000,Mars 2025



Top 5 des postes pesant le plus sur la masse salariale :


3,Libellé de poste,Country,Effectif,Budget_Total_Poste,Période
307,CLIENT OPERATIONS OFFICER,France,325,15886000,Mars 2025
704,BUSINESS COORDINATOR,Luxembourg,135,13026150,Mars 2025
830,TEAM MANAGER,Luxembourg,101,11005869,Mars 2025
343,FUND ACCOUNTANT,France,234,10503324,Mars 2025
714,CLIENT OPERATIONS OFFICER,Luxembourg,155,9675100,Mars 2025



Top 5 des postes avec la plus forte part de variable (%) :


3,Libellé de poste,Country,Pct_Variable,Période
732,Deputy CEO,Luxembourg,58.725897,Mars 2025
299,CHIEF FINANCIAL AND ADMINISTRATIVE OFFICER,France,52.380952,Mars 2025
323,DEPUTY CHIEF EXECUTIVE OFFICER,France,52.054795,Mars 2025
723,COUNTRY MANAGING DIRECTOR,Luxembourg,48.684070,Mars 2025
300,CHIEF HUMAN RESOURCES OFFICER,France,48.314607,Mars 2025


In [30]:
# Agrégation par pays (basée sur la période la plus récente : Mars 2025)
# On exclut d'abord les lignes de totaux pour ne pas doubler les chiffres
df_clean = df_data_salaire[~df_data_salaire['Libellé de poste'].str.contains('Total', case=False, na=False)]
latest_data = df_clean[df_clean['Période'] == 'Mars 2025']

# Calcul de la masse salariale totale par ligne avant agrégation
latest_data['Masse_Salariale_Poste'] = latest_data['Total_Comp_Indiv'] * latest_data['Effectif']

# Agrégation finale
stats_pays = latest_data.groupby('Country').agg({
    'Effectif': 'sum',
    'Masse_Salariale_Poste': 'sum'
}).reset_index()

# Calcul du coût moyen par tête par pays (Insight supplémentaire)
stats_pays['Cout_Moyen_Par_Tete'] = (stats_pays['Masse_Salariale_Poste'] / stats_pays['Effectif']).astype(int)

# On définit un dictionnaire de formatage
format_dict = {
    'Effectif': '{:,.0f}',
    'Masse_Salariale_Poste': '{:,.0f} €',
    'Cout_Moyen_Par_Tete': '{:,.0f} €'
}

print("--- ANALYSE PAR PAYS (Mars 2025) ---")
styled_stats = stats_pays.style.format(format_dict, thousands=" ")

# Affichage
display(styled_stats)

--- ANALYSE PAR PAYS (Mars 2025) ---


C:\Users\Dalanda\AppData\Local\Temp\ipykernel_14284\2524755751.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  latest_data['Masse_Salariale_Poste'] = latest_data['Total_Comp_Indiv'] * latest_data['Effectif']


3,Country,Effectif,Masse_Salariale_Poste,Cout_Moyen_Par_Tete
0,France,1 970,132 411 171 €,67 213 €
1,Luxembourg,1 778,179 247 350 €,100 814 €


### Training

In [39]:
def process_training_data(file_path):
    # 1. CHARGEMENT
    # Chargement de l'onglet spécifique
    df_training = pd.read_excel(file_path, sheet_name='Final_CSV')
    
    print("--- 1. STRUCTURE ET NETTOYAGE ---")
    # Affichage des dimensions et types
    print(f"Dimensions : {df_training.shape[0]} lignes, {df_training.shape[1]} colonnes")
    
    # Conversion des dates pour le Machine Learning
    date_cols = ['Seesion_Start_Date', 'Session_End_Date']
    for col in date_cols:
        df_training[col] = pd.to_datetime(df_training[col], errors='coerce')
    
    # Nettoyage des colonnes numériques (Total Training Hours)
    df_training['Total_Training_Hours'] = pd.to_numeric(df_training['Total_Training_Hours'], errors='coerce').fillna(0).astype(int)
    
    # Affichage des types après conversion
    display(df_training.dtypes.to_frame(name='Type de données'))
    
    # Analyse des valeurs manquantes
    missings = df_training.isnull().sum().to_frame(name='Valeurs Manquantes')
    display(missings[missings['Valeurs Manquantes'] > 0])
    
    print("\n--- 2. STATISTIQUES DESCRIPTIVES ---")
    # Statistiques sur les heures de formation et les certifications
    # On inclut 'include='all'' pour voir aussi les directions et statuts les plus fréquents
    display(df_training.describe(include='all'))
    
    return df_training


In [40]:
# Utilisation
df_training_final = process_training_data("Training_Records_Unnamed.xlsx")

--- 1. STRUCTURE ET NETTOYAGE ---
Dimensions : 14943 lignes, 12 colonnes


C:\Users\Dalanda\AppData\Local\Temp\ipykernel_14284\3052982967.py:13: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df_training[col] = pd.to_datetime(df_training[col], errors='coerce')
C:\Users\Dalanda\AppData\Local\Temp\ipykernel_14284\3052982967.py:13: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df_training[col] = pd.to_datetime(df_training[col], errors='coerce')


,Type de données
Employee Code,object
Entity,object
Direction,object
Attended_Courses,object
Organization,object
Seesion_Start_Date,datetime64[ns]
Session_End_Date,datetime64[ns]
Session_ID,object
Status,object
Total_Training_Hours,int64


,Valeurs Manquantes
Employee Code,443
Entity,30
Direction,442
Organization,1
Seesion_Start_Date,454
Session_End_Date,454
Session_ID,1625
Certifications,5975



--- 2. STATISTIQUES DESCRIPTIVES ---


,Employee Code,Entity,Direction,Attended_Courses,Organization,Seesion_Start_Date,Session_End_Date,Session_ID,Status,Total_Training_Hours,Certifications,Year
count,14500,14913,14501,14943,14942,14489,14489,13318,14943,14943.000000,8968,14943.000000
unique,2482,4,41,767,207,NaN,NaN,1292,5,NaN,3,NaN
top,ANON_66X72X48X48X48X48X51X53X48X52X56X,CACEIS Bank,BUT - Fund Services,Réunion d'information MyDev - Collaborateurs,INTERNE,NaN,NaN,23-RDM-221-456,Réalisé,NaN,No,NaN
freq,40,10974,2974,708,2589,NaN,NaN,343,13446,NaN,8815,NaN
mean,NaN,NaN,NaN,NaN,NaN,2024-05-12 10:49:47.038443264,2024-07-09 07:39:09.727379456,NaN,NaN,8.071271,NaN,2023.884361
min,NaN,NaN,NaN,NaN,NaN,2022-09-29 00:00:00,2022-09-30 00:00:00,NaN,NaN,0.000000,NaN,2023.000000
25%,NaN,NaN,NaN,NaN,NaN,2023-09-11 00:00:00,2023-11-23 00:00:00,NaN,NaN,1.000000,NaN,2023.000000
50%,NaN,NaN,NaN,NaN,NaN,2024-04-09 00:00:00,2024-05-23 00:00:00,NaN,NaN,4.000000,NaN,2024.000000
75%,NaN,NaN,NaN,NaN,NaN,2025-02-04 00:00:00,2025-03-11 00:00:00,NaN,NaN,14.000000,NaN,2025.000000
max,NaN,NaN,NaN,NaN,NaN,2026-12-31 00:00:00,2026-12-31 00:00:00,NaN,NaN,1008.000000,NaN,2025.000000


In [41]:
# --- KEY INSIGHT EXPLORATOIRE (Formations Réalisées) ---

def analyze_realized_training(df_training):
    print("\n--- ANALYSE DES FORMATIONS RÉALISÉES ---")
    
    # 1. Filtrage sur le statut "Réalisé"
    df_realized = df_training[df_training['Status'] == 'Réalisé'].copy()
    
    # 2. Agrégation par Direction
    # On compte les heures totales et le nombre d'employés uniques (Employee Code)
    direction_stats = df_realized.groupby('Direction').agg({
        'Total_Training_Hours': 'sum',
        'Employee Code': 'nunique'  # nunique pour compter les personnes physiques sans doublons
    }).reset_index()
    
    # Renommer pour plus de clarté
    direction_stats.columns = ['Direction', 'Total_Heures', 'Nombre_Salaries_Formes']
    
    # 3. Top 5 par Heures de Training
    print("\nTop 5 des Directions - Plus grand volume d'heures (Réalisé) :")
    top_hours = direction_stats.sort_values(by='Total_Heures', ascending=False).head(5)
    display(top_hours.style.format({'Total_Heures': '{:,.0f}'}, thousands=" "))
    
    # 4. Top 5 par Nombre de Personnes en formation
    print("\nTop 5 des Directions - Plus grand nombre de salariés formés (Réalisé) :")
    top_employees = direction_stats.sort_values(by='Nombre_Salaries_Formes', ascending=False).head(5)
    display(top_employees.style.format({'Nombre_Salaries_Formes': '{:,.0f}'}, thousands=" "))

    return top_hours, top_employees

In [42]:
# Utilisation :
top_h, top_e = analyze_realized_training(df_training_final)


--- ANALYSE DES FORMATIONS RÉALISÉES ---

Top 5 des Directions - Plus grand volume d'heures (Réalisé) :


,Direction,Total_Heures,Nombre_Salaries_Formes
3,BUT - Fund Services,24 419,628
2,BUT - Custody & Cash Clearing,11 619,424
14,COV - PERES,10 845,255
6,BUT - Information Technology,7 690,161
30,SPF - Corporate Compliance,5 667,61



Top 5 des Directions - Plus grand nombre de salariés formés (Réalisé) :


,Direction,Total_Heures,Nombre_Salaries_Formes
3,BUT - Fund Services,24 419,628
2,BUT - Custody & Cash Clearing,11 619,424
14,COV - PERES,10 845,255
6,BUT - Information Technology,7 690,161
10,COV - Client & Bus Dev Support,5 651,119


In [43]:
# Filtrage sur le statut "Réalisé"
df_realized = df_training_final[df_training_final['Status'] == 'Réalisé'].copy()

# Calcul du nombre total de salariés uniques
total_salaries_formes = df_realized['Employee Code'].nunique()

print(f"--- RÉSULTAT GLOBAL ---")
print(f"Nombre total de salariés ayant réalisé au moins une formation : {total_salaries_formes}")

# Optionnel : Si tu veux le détail par Entité (CACEIS Bank FR vs LU)
salaries_par_entite = df_realized.groupby('Entity')['Employee Code'].nunique().reset_index()
salaries_par_entite.columns = ['Entité', 'Nb Salariés Formés']

display(salaries_par_entite)

--- RÉSULTAT GLOBAL ---
Nombre total de salariés ayant réalisé au moins une formation : 2447


,Entité,Nb Salariés Formés
0,CACEIS,113
1,CACEIS Bank,1693
2,CACEIS Fund Administration,636
3,CACEIS Sa,0


### Notes et évaluation (EAE EP)

In [54]:
def process_eae_full(file_path):
    # 1. CHARGEMENT
    df_eae = pd.read_excel(file_path)
    
    # Nettoyage des noms de colonnes
    df_eae.columns = df_eae.columns.astype(str).str.strip()

    print("--- 1. STRUCTURE, TYPES ET VALEURS MANQUANTES ---")
    # Création du tableau récapitulatif avec les cellules vides
    info_df = pd.DataFrame({
        'Type de données': df_eae.dtypes,
        'Cellules Vides (NaN)': df_eae.isnull().sum(),
        '% de Vide': (df_eae.isnull().sum() / len(df_eae) * 100).round(2)
    })
    display(info_df)
    print(f"Total : {df_eae.shape[0]} lignes | {df_eae.shape[1]} colonnes")

    # Traitement des colonnes clés
    if 'IUG' in df_eae.columns:
        df_eae['IUG'] = df_eae['IUG'].astype(str)
    
    if 'Note de performance' in df_eae.columns:
        # Extraction du score numérique (ex: "3 FR" -> 3)
        df_eae['Note_Numeric'] = pd.to_numeric(
            df_eae['Note de performance'].astype(str).str.extract('(\d+)', expand=False), 
            errors='coerce'
        )
    
    # --- 2. ANALYSE STATISTIQUE ---
    print("\n--- 2. ANALYSE STATISTIQUE DES PERFORMANCES ---")
    
    if 'Note_Numeric' in df_eae.columns:
        stats_notes = df_eae['Note_Numeric'].describe().to_frame(name='Valeurs')
        if not df_eae['Note_Numeric'].mode().empty:
            stats_notes.loc['mode'] = df_eae['Note_Numeric'].mode()[0]
        display(stats_notes)
        
        # Analyse par Country (Utilisation directe sans mapping)
        if 'Country' in df_eae.columns:
            print("\nComparaison de la performance par Country :")
            stats_pays = df_eae.groupby('Country')['Note_Numeric'].agg(['mean', 'std', 'count', 'median']).round(2)
            display(stats_pays)

    # C. Taux de complétion
    if 'Statut du document' in df_eae.columns:
        print("\nRépartition des statuts d'entretiens (en %) :")
        statut_counts = df_eae['Statut du document'].value_counts(normalize=True).to_frame(name='%') * 100
        display(statut_counts.round(2))

    # D. Matrice de corrélation
    print("\nMatrice de corrélation (Variables numériques) :")
    display(df_eae.select_dtypes(include=[np.number]).corr())

    return df_eae

In [55]:
# Utilisation
df_eae_final = process_eae_full("20250218 - Stats CACEIS EAE EP 18-02-2025 Version Définitive cloture.xlsx")

--- 1. STRUCTURE, TYPES ET VALEURS MANQUANTES ---


,Type de données,Cellules Vides (NaN),% de Vide
BU,object,0,0.00
Année,int64,0,0.00
Nom du document,object,0,0.00
Statut du document,object,0,0.00
Evaluation manager,object,2289,39.70
IUG,object,4,0.07
Nom,object,0,0.00
Prénom,object,0,0.00
Mail collaborateur,object,0,0.00
Libellé emploi,object,3,0.05


Total : 5766 lignes | 32 colonnes

--- 2. ANALYSE STATISTIQUE DES PERFORMANCES ---


,Valeurs
count,3477.000000
mean,3.391717
std,0.623310
min,1.000000
25%,3.000000
50%,3.000000
75%,4.000000
max,5.000000
mode,3.000000



Comparaison de la performance par Country :


,mean,std,count,median
Country,,,,
France,3.35,0.62,1849,3.0
Luxembourg,3.44,0.62,1628,3.0



Répartition des statuts d'entretiens (en %) :


,%
Statut du document,
Entretien(s) terminé(s),88.78
Auto-évaluation(s) collaborateur en cours,7.44
Signature(s) en cours collaborateur,3.00
Entretiens(s) en cours manager,0.69
Entretien(s) annulé(s),0.09



Matrice de corrélation (Variables numériques) :


,Année,Code organisation niveau 08,Libellé Organisation niveau 08,Code organisation niveau 09,Libellé Organisation niveau 09,Code organisation niveau 10,Libellé Organisation niveau 10,Note de performance,Unnamed: 30,Note_Numeric
Année,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Code organisation niveau 08,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Libellé Organisation niveau 08,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Code organisation niveau 09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Libellé Organisation niveau 09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Code organisation niveau 10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Libellé Organisation niveau 10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Note de performance,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,1.0
Unnamed: 30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Note_Numeric,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,1.0


In [56]:
# --- ANALYSE DE LA PERFORMANCE PAR DIRECTION (NIVEAU 05) ---

def analyze_perf_by_direction(df_eae):
    print("\n--- PERFORMANCE MOYENNE PAR DIRECTION (Livrable 05) ---")
    
    # On vérifie que la colonne et la note numérique existent
    col_dir = 'Libellé Organisation niveau 05'
    
    if col_dir in df_eae.columns and 'Note_Numeric' in df_eae.columns:
        # Agrégation : Moyenne, Ecart-type (pour voir la disparité) et effectif
        perf_dir = df_eae.groupby(col_dir)['Note_Numeric'].agg([
            'mean', 
            'std', 
            'count'
        ]).reset_index()
        
        # Renommage pour plus de clarté
        perf_dir.columns = ['Direction', 'Note_Moyenne', 'Dispersion_Notes', 'Nb_Salaries']
        
        # Tri par note moyenne décroissante
        perf_dir = perf_dir.sort_values(by='Note_Moyenne', ascending=False)
        
        # Affichage avec formatage
        display(perf_dir.style.format({
            'Note_Moyenne': '{:.2f}',
            'Dispersion_Notes': '{:.2f}'
        }).background_gradient(subset=['Note_Moyenne'], cmap='RdYlGn')) # Vert = bon, Rouge = bas
        
        return perf_dir
    else:
        print(f"⚠️ La colonne '{col_dir}' est absente du fichier.")


In [57]:
# Utilisation :
df_perf_directions = analyze_perf_by_direction(df_eae_final)


--- PERFORMANCE MOYENNE PAR DIRECTION (Livrable 05) ---


,Direction,Note_Moyenne,Dispersion_Notes,Nb_Salaries
9,COV - Coverage,4.00,nan,1
14,HUMAN RESOURCES,4.00,nan,1
11,COV - Coverage France,3.67,0.82,42
26,STI - Communications,3.63,0.50,19
22,SPF - Procurement,3.62,0.52,8
24,STI - 3D & Products,3.55,0.69,38
10,COV - Coverage Europe excl Fr,3.53,0.73,38
4,BUT - Inf System Sec & Resil,3.52,0.81,21
6,BUT - Market Solutions,3.50,0.62,234
12,COV - PERES,3.49,0.65,373


### Croisement des fichiers

In [58]:
# --- CROISEMENT PERFORMANCE (EAE) vs FORMATION (TRAINING) ---

def analyze_training_impact_on_perf(df_eae, df_training):
    print("\n--- ANALYSE : IMPACT DE LA FORMATION SUR LA PERFORMANCE ---")
    
    # 1. Préparation des données EAE (Moyenne par Direction)
    # On utilise 'Libellé Organisation niveau 05'
    perf_by_dir = df_eae.groupby('Libellé Organisation niveau 05')['Note_Numeric'].mean().reset_index()
    perf_by_dir.columns = ['Direction_Key', 'Note_Moyenne']
    
    # 2. Préparation des données Training (Moyenne d'heures par Direction)
    # On utilise 'Direction' et on filtre sur 'Réalisé'
    training_realized = df_training[df_training['Status'] == 'Réalisé'].copy()
    training_by_dir = training_realized.groupby('Direction')['Total_Training_Hours'].mean().reset_index()
    training_by_dir.columns = ['Direction_Key', 'Heures_Formation_Moyennes']
    
    # 3. Jointure (Merge) entre les deux
    # On aligne 'Libellé Organisation niveau 05' et 'Direction'
    df_impact = pd.merge(perf_by_dir, training_by_dir, on='Direction_Key', how='inner')
    
    # 4. Calcul de la corrélation
    correlation = df_impact['Note_Moyenne'].corr(df_impact['Heures_Formation_Moyennes'])
    
    # 5. Affichage du Top des Directions
    print(f"Corrélation globale entre Formation et Performance : {correlation:.2f}")
    
    # Tri pour voir les directions les plus "apprenantes" et leurs notes
    df_impact_sorted = df_impact.sort_values(by='Heures_Formation_Moyennes', ascending=False)
    
    display(df_impact_sorted.style.background_gradient(cmap='Blues', subset=['Heures_Formation_Moyennes'])
                              .background_gradient(cmap='Greens', subset=['Note_Moyenne']))
    
    return df_impact_sorted

In [59]:
# Utilisation :
df_correlation = analyze_training_impact_on_perf(df_eae_final, df_training_final)


--- ANALYSE : IMPACT DE LA FORMATION SUR LA PERFORMANCE ---
Corrélation globale entre Formation et Performance : -0.64


,Direction_Key,Note_Moyenne,Heures_Formation_Moyennes
0,BUT - Business Units & Tech,3.000000,12.666667
23,STI - ESG,2.800000,10.285714
8,COV - Client Success,3.338462,9.899306
21,STI - CACEIS Consulting,3.340909,9.420485
3,BUT - Gen Secretary & Controls,3.352941,8.994872
4,BUT - Inf System Sec & Resil,3.523810,8.980892
2,BUT - Fund Services,3.341637,8.954529
14,SPF - Corporate Compliance,3.383562,8.799689
5,BUT - Information Technology,3.361823,8.738636
9,COV - Coverage France,3.666667,8.651163


Notre analyse exploratoire révèle un paradoxe : les directions ayant le plus fort volume de formation affichent les notes de performance les plus basses (Corrélation : -0.64). Cela suggère que la formation est actuellement utilisée comme un outil de remédiation pour les secteurs en difficulté plutôt que comme un levier de développement pour les hauts potentiels. 